In [1]:
import tiktoken
import re
import json
from collections import Counter, deque
from functools import lru_cache
import requests

%load_ext autoreload
%autoreload 2

from bpe import BPETokenizerSimple

In [2]:
import os
import requests

main_dir='/Users/reaungamornrat.sureerat/data/BPE/pracetice'
file_path=f"{main_dir}/the-verdict.txt"

if not os.path.exists(file_path):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    response = requests.get(url, verify=False)
    with open(file_path, 'wb') as f: f.write(response.content)

with open(file_path, "r", encoding="utf-8") as f:  text = f.read()

In [3]:
def download_file_if_absent(url, filename, dirpath):
    """
    Args:
        dirpath (str):Directory to search for file and save file if absent 
    Returns:
        (str): File location
    """
    filepath=os.path.join(dirpath, filename)
    if os.path.exists(filepath):
        print(f"{filepath} already exists in {dirpath}")
        return filepath

    response=requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    with open(filepath, "wb") as out_file:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk: out_file.write(chunk)
    print(f"Download {filename} to {filepath}")
    return filepath
    
# loadng the original GPT-2 BPE tokenizer from openai
files_to_download = {
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe": "vocab.bpe",
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json": "encoder.json"
}
gpt2_dir=f"{main_dir}/gpt2_bpe"
if not os.path.isdir(gpt2_dir): os.makedirs(gpt2_dir)

paths={}
for url, filename in files_to_download.items():
    paths[filename]=download_file_if_absent(url, filename, gpt2_dir)

/Users/reaungamornrat.sureerat/data/BPE/pracetice/gpt2_bpe/vocab.bpe already exists in /Users/reaungamornrat.sureerat/data/BPE/pracetice/gpt2_bpe
/Users/reaungamornrat.sureerat/data/BPE/pracetice/gpt2_bpe/encoder.json already exists in /Users/reaungamornrat.sureerat/data/BPE/pracetice/gpt2_bpe


In [4]:
replace_space=None# "Ġ"

tokenizer=BPETokenizerSimple(replace_space=replace_space)
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})
print(f"{len(tokenizer.vocab)=}, {len(tokenizer.bpe_merges)=}")
print(list(tokenizer.vocab.items())[-100:])

len(tokenizer.vocab)=1000, len(tokenizer.bpe_merges)=681
[(900, ' where'), (901, 'air'), (902, ' head'), (903, ' looking'), (904, ' mov'), (905, ' manag'), (906, ' let'), (907, ' hal'), (908, ' half'), (909, 'rself'), (910, ' sw'), (911, ' ti'), (912, 'sw'), (913, ' began'), (914, ' still'), (915, ' long'), (916, ' min'), (917, 'pect'), (918, 'pected'), (919, ' us'), (920, ' never'), (921, 'ash'), (922, ' sket'), (923, ' sketch'), (924, ' beh'), (925, ' behind'), (926, ' done'), (927, ' poor'), (928, ' me.'), (929, ' tw'), (930, ' off'), (931, ' watching'), (932, ' fell'), (933, 'pped'), (934, ' marr'), (935, ' married'), (936, 'ich'), (937, ' Riv'), (938, ' Rivi'), (939, ' Rivier'), (940, ' Riviera'), (941, ' ('), (942, ' F'), (943, '--that'), (944, ' call'), (945, ' called'), (946, ' un'), (947, " it's"), (948, ' lips,'), (949, 'fle'), (950, ' Her'), (951, ' before'), (952, ' "W'), (953, ' look'), (954, 'oor'), (955, ' made'), (956, ' heard'), (957, ' har'), (958, ' hard'), (959, ' h

In [9]:
replace_space="Ġ"

tokenizer=BPETokenizerSimple(replace_space=replace_space)
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})
print(f"{len(tokenizer.vocab)=}, {len(tokenizer.bpe_merges)=}")
print(list(tokenizer.vocab.items())[-100:])

len(tokenizer.vocab)=1000, len(tokenizer.bpe_merges)=681
[(900, 'Ġwhere'), (901, 'air'), (902, 'Ġhead'), (903, 'Ġlooking'), (904, 'Ġmov'), (905, 'Ġmanag'), (906, 'Ġlet'), (907, 'Ġhal'), (908, 'Ġhalf'), (909, 'rself'), (910, 'Ġsw'), (911, 'Ġti'), (912, 'sw'), (913, 'Ġbegan'), (914, 'Ġstill'), (915, 'Ġlong'), (916, 'Ġmin'), (917, 'pect'), (918, 'pected'), (919, 'Ġus'), (920, 'Ġnever'), (921, 'ash'), (922, 'Ġsket'), (923, 'Ġsketch'), (924, 'Ġbeh'), (925, 'Ġbehind'), (926, 'Ġdone'), (927, 'Ġpoor'), (928, 'Ġme.'), (929, 'Ġtw'), (930, 'Ġoff'), (931, 'Ġwatching'), (932, 'Ġfell'), (933, 'pped'), (934, 'Ġmarr'), (935, 'Ġmarried'), (936, 'ich'), (937, 'ĠRiv'), (938, 'ĠRivi'), (939, 'ĠRivier'), (940, 'ĠRiviera'), (941, 'Ġ('), (942, 'ĠF'), (943, '--that'), (944, 'Ġcall'), (945, 'Ġcalled'), (946, 'Ġun'), (947, "Ġit's"), (948, 'Ġlips,'), (949, 'fle'), (950, 'ĠHer'), (951, 'Ġbefore'), (952, 'Ġ"W'), (953, 'Ġlook'), (954, 'oor'), (955, 'Ġmade'), (956, 'Ġheard'), (957, 'Ġhar'), (958, 'Ġhard'), (959, 'Ġh

In [12]:
input_text="Jack embraced beauty through art and life."
token_ids=tokenizer.encode(input_text, allowed_special=None)
print(token_ids)

input_text="Jack embraced beauty through art and life.<|endoftext|>"
token_ids=tokenizer.encode(input_text, allowed_special={"<|endoftext|>"})
print(token_ids)

print(f"\nNumber of characters: {len(input_text)}")
print(f"Number of token IDs: {len(token_ids)}")

[278, 422, 371, 304, 293, 481, 458, 295, 361, 489, 311, 316, 580, 760, 360, 869, 595]
[278, 422, 371, 304, 293, 481, 458, 295, 361, 489, 311, 316, 580, 760, 360, 869, 595, 318]

Number of characters: 55
Number of token IDs: 18


In [19]:
print(tokenizer.decode(token_ids))
# Most token IDs represent 2-character subwords because the training data text is very short with small repetitive words
# and because we used a relatively small vocabulary size
for token_id in token_ids: print(f"{token_id} -> {tokenizer.decode([token_id])}")

Jack embraced beauty through art and life.<|endoftext|>
278 -> J
422 -> ack
371 ->  e
304 -> m
293 -> b
481 -> ra
458 -> ce
295 -> d
361 ->  be
489 -> au
311 -> t
316 -> y
580 ->  through
760 ->  art
360 ->  and
869 ->  lif
595 -> e.
318 -> <|endoftext|>


In [22]:
print(f"{tokenizer.decode(tokenizer.encode('This is some text'))=}")
print(tokenizer.decode(tokenizer.encode('This is some text with \n newline characters')))

tokenizer.decode(tokenizer.encode('This is some text'))='This is some text'
This is some text with 
 newline characters


In [26]:
dirpath=f"{main_dir}/custom_bpe"
if not os.path.isdir(dirpath): os.makedirs(dirpath)
vocab_path=f"{dirpath}/vocab.json"
bpe_merges_path=f"{dirpath}/bpe_merges.txt"

# save trained tokenizer
tokenizer.save_vocab_and_merges(vocab_path=vocab_path, bpe_merges_path=bpe_merges_path)

In [37]:
tokenizer2=BPETokenizerSimple(replace_space=None)
tokenizer2.load_vocab_and_merges(vocab_path=vocab_path, bpe_merges_path=bpe_merges_path)
print(tokenizer2.decode(token_ids))
tokenizer2.decode(tokenizer2.encode("This is some text with \n newline characters."))

Jack embraced beauty through art and life.<|endoftext|>


'This is some text with \n newline characters.'

### Load the original GPT-2 BPE tokenizer from OpenAI

In [41]:
tokenizer_gpt2=BPETokenizerSimple(replace_space="Ġ")
tokenizer_gpt2.load_vocab_and_merge_from_openai(vocab_path=paths['encoder.json'], bpe_merges_path=paths['vocab.bpe'])
print(f"{len(tokenizer_gpt2.vocab)=}")

len(tokenizer_gpt2.vocab)=50257


In [42]:
input_text="This is some text"
token_ids=tokenizer_gpt2.encode(input_text)
print(f"{token_ids=}")
print(tokenizer_gpt2.decode(token_ids))

token_ids=[1212, 318, 617, 2420]
This is some text


In [43]:
gpt2_tokenizer=tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode(input_text)

[1212, 318, 617, 2420]